# Lab 1 — Spectral Analysis of Deterministic Signals

**All 15 variants** | Python 3 / Jupyter Notebook

## Objective
For every coefficient vector in the laboratory sheet, construct the $N\times N$ index-product matrix $K$ and Fourier synthesis matrix $W$, perform an IDFT using matrix multiplication, show the matrices, plot the synthesized time-domain signal and independently validate the result.

Some spectra are not conjugate-symmetric, so their time-domain signals can be complex. We plot real, imaginary and magnitude components.

## Mathematics

$$K_{k,\mu}=k\mu,\quad W_{k,\mu}=e^{+j2\pi k\mu/N},\quad x[k]=\frac{1}{N}\sum_{\mu=0}^{N-1}X[\mu]W_{k,\mu}.$$

Equivalently, $\mathbf{x}=(W\mathbf{X})/N$. Matrix orthogonality: $W^HW=NI$. The forward DFT uses $W^*\mathbf{x}$. `numpy.fft.ifft` is used for verification, **not for synthesis**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True, linewidth=130)
VARIANTS = {"1":[6,2,4,3,4,5,0,0,0,0],"2":[10,5,6,6,2,4,3,4,5,0,0,0,0],"3":[6,2,4,3,4,5,0,0,0,0],"4":[6,2,4,3,4,5,0,0,0],"5":[6,4,4,5,3,4,5,0,0,0,0],"6":[7,2,4,3,4,5,0,0,0,0],"7":[6,8,2,4,3,4,5,0,0,0],"8":[6,2,4,4,4,5,0,0,0,0],"9":[6,5,4,3,4,5,0,0,0,0],"10":[6,2,4,3,4,4,0,0,0,0],"11":[6,2,4,3,4,5,0,0,0,0],"12":[10,5,6,6,2,4,3,4,5,0,0,0,0],"13":[6,2,4,3,4,5,0,0,0,0],"14":[6,2,4,3,4,5,0,0,0],"15":[6,4,4,5,3,4,5,0,0,0,0]}
print('Variants:', len(VARIANTS))

In [ ]:
def fourier_matrices(N):
    """Return K[k, mu]=k*mu and the positive-exponent Fourier matrix."""
    n = np.arange(N)
    K = np.outer(n, n)
    W = np.exp(2j * np.pi * K / N)
    return K, W

def synthesize(spectrum):
    """Matrix IDFT and independent verification."""
    X = np.asarray(spectrum, dtype=complex)
    N = len(X)
    K, W = fourier_matrices(N)
    x = W @ X / N
    error = np.max(np.abs(x - np.fft.ifft(X)))
    dft_error = np.max(np.abs(np.conjugate(W) @ x - X))
    assert error < 1e-12, "IDFT mismatch"
    assert dft_error < 1e-12, "DFT reconstruction mismatch"
    assert np.allclose(W.conj().T @ W, N*np.eye(N), atol=1e-12)
    return {"N": N, "X": X, "K": K, "W": W, "x": x,
            "idft_error": error, "dft_error": dft_error}

def plot_signal(x, variant):
    k = np.arange(len(x))
    fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True, constrained_layout=True)
    for ax, value, title in zip(axs, (x.real, x.imag, np.abs(x)),
                                 ("Real part Re{x[k]}", "Imaginary part Im{x[k]}",
                                  "Magnitude |x[k]|")):
        ax.stem(k, value, basefmt="k-")
        ax.set_ylabel(title)
        ax.grid(alpha=.3)
    axs[-1].set_xlabel("Sample index k")
    axs[-1].set_xticks(k)
    fig.suptitle(f"Variant {variant} — IDFT synthesis (N={len(x)})")
    plt.show()


## All variants

For every numbered variant, the following code displays the complete $K$ and $W$ matrices (the latter rounded only for display), all synthesized samples and three graphs. The original 15 coefficient vectors are preserved exactly as in the laboratory PDF.

In [ ]:
RESULTS = {}
for variant, coeffs in VARIANTS.items():
    r = synthesize(coeffs)
    RESULTS[variant] = r
    print("=" * 110)
    print(f"VARIANT {variant} | N={r['N']} | X={coeffs}")
    print("K =")
    print(r["K"])
    print("W = (3 decimal places for display)")
    print(np.array2string(r["W"], precision=3, suppress_small=True))
    print("x[k] =")
    print(np.array2string(r["x"], precision=5, suppress_small=True))
    print(f"Max IDFT error: {r['idft_error']:.3e}")
    print(f"Max forward DFT error: {r['dft_error']:.3e}")
    print(f"Real-valued signal: {np.allclose(r['x'].imag, 0, atol=1e-12)}")
    plot_signal(r["x"], variant)
print("PASS: All 15 variants synthesized and verified.")

## Summary and conclusions

The Fourier synthesis matrix implements IDFT directly by matrix multiplication. The original spectrum is reconstructed using the forward DFT, and the results agree with NumPy's IDFT to floating-point precision. Non-conjugate-symmetric spectra generally produce complex-valued signals, so both real and imaginary parts are shown. Identical input spectra produce identical outputs but are still treated as separately numbered variants. The sample mean equals $X[0]/N$.

**Source:** `Lab1_EN.pdf`, Section 7, variants 1–15.